# Caminhos UFV — análise reproduzível do instrumento 4.2
Pesquisa de doutorado sobre mobilidade em Viçosa. Este caderno executa a análise do questionário revisado. A base inicial contém 400 registros fictícios, sem participantes humanos. A rotina não usa os construtos nem os nove cenários da versão 3.

Escolha a origem dos dados na célula de configuração. Não salve saídas de microdados reais em um caderno público. O código implementa descrição, aprendizado de máquina com validação interna, GEE para seis respostas repetidas e interação exploratória. Mediação causal não é identificada por esta coleta transversal.

In [ ]:
%pip -q install numpy==2.5.4 pandas==3.0.6 scipy==1.18.1 scikit-learn==1.9.1 statsmodels==0.15.0 matplotlib networkx folium gspread google-auth

In [ ]:
from pathlib import Path
import json, hashlib, urllib.request, importlib.util
import numpy as np, pandas as pd
SOURCE = 'synthetic'  # 'synthetic', 'upload' ou 'google_sheets'
SHEET_ID = 'PREENCHA_O_ID_DA_SUA_PLANILHA_PRIVADA'
SHEET_TAB = 'respostas'  # para simulação integrada: simulacao_v4
BASE = 'https://caminhos-ufv.vercel.app/'
OUTPUT = Path('resultados_ufv_v4'); OUTPUT.mkdir(exist_ok=True)


In [ ]:
if SOURCE == 'synthetic':
    with urllib.request.urlopen(BASE+'respostas-ficticias-v4-400.json') as response:
        data = json.load(response)
    records = data['records']
elif SOURCE == 'upload':
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1: raise ValueError('Envie um único arquivo JSON da administração.')
    data = json.loads(next(iter(uploaded.values())))
    records = data if isinstance(data,list) else data.get('records',[data])
elif SOURCE == 'google_sheets':
    from google.colab import auth
    import google.auth, gspread
    auth.authenticate_user()
    credentials,_ = google.auth.default()
    book = gspread.authorize(credentials).open_by_key(SHEET_ID)
    values = book.worksheet(SHEET_TAB).get_all_values()
    if not values: raise ValueError('Aba vazia.')
    header = values[0]
    idx = header.index('response_json') if 'response_json' in header else 2
    records = [json.loads(row[idx]) for row in values[1:] if len(row)>idx and row[idx]]
else:
    raise ValueError('Fonte desconhecida.')
versions = pd.Series([r.get('schema_version') for r in records]).value_counts()
print('Versões encontradas:', versions.to_dict())
if any(r.get('schema_version') != 'ufv-mobilidade-4.2.0' for r in records):
    raise ValueError('Não misture versões. Exporte o instrumento 4.2 pela administração.')
if len({r['response_id'] for r in records}) != len(records): raise ValueError('Identificadores duplicados.')
kinds = {('synthetic' if r['is_synthetic'] else 'demo' if r['is_demo'] else 'real') for r in records}
if len(kinds) != 1: raise ValueError('Separe dados reais e sintéticos.')
print('Registros:',len(records),'Tipo:',next(iter(kinds)))


## Código integral dos modelos
A célula seguinte contém a implementação utilizada pelo endpoint do painel. Não é uma chamada a uma caixa-preta. As funções podem ser inspecionadas e modificadas. Preditores são tratados dentro de cada partição de treino; qChance não entra em qChange por depender da própria resposta ao desfecho. Os modelos usam hiperparâmetros fixados, sem busca no conjunto de teste.

In [ ]:
Path('api').mkdir(exist_ok=True); Path('public').mkdir(exist_ok=True)
Path('api/analysis_v4.py').write_text('"""Analysis for the revised UFV questionnaire. No cross-version recoding."""\nimport os,json,hashlib,hmac,base64,time,warnings\nfrom http.server import BaseHTTPRequestHandler\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport statsmodels.api as sm\nimport statsmodels.formula.api as smf\nfrom sklearn.model_selection import StratifiedKFold\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.preprocessing import OneHotEncoder,StandardScaler\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.dummy import DummyClassifier\nfrom sklearn.metrics import balanced_accuracy_score,roc_auc_score,brier_score_loss,log_loss\nSEED=8421264\nVERSION=\'ufv-mobilidade-4.2.0\'\n\ndef clean(v):\n if isinstance(v,dict):return {str(k):clean(x) for k,x in v.items()}\n if isinstance(v,(list,tuple,np.ndarray)):return [clean(x) for x in v]\n if isinstance(v,(np.integer,)):return int(v)\n if isinstance(v,(float,np.floating)):return float(v) if np.isfinite(v) else None\n return v\n\ndef frame(records):\n rows=[]\n for r in records:\n  a=r[\'answers\'];choice=a.get(\'qChange\')\n  rows.append(dict(id=r[\'response_id\'],target=int(choice.startswith(\'Sim\')) if isinstance(choice,str) else np.nan,age=a.get(\'q2\'),role=a.get(\'q3\'),income=a.get(\'q5\'),housing=a.get(\'qHousing\'),skill=a.get(\'q10\'),mode=a.get(\'q16\')))\n return pd.DataFrame(rows).replace({None:np.nan})\n\ndef ml(records):\n df=frame(records).dropna(subset=[\'target\']);y=df.target.astype(int).to_numpy();counts=pd.Series(y).value_counts()\n if len(df)<80 or len(counts)!=2 or counts.min()<20:raise ValueError(\'ML requer 80 respostas válidas e pelo menos 20 em cada classe de disposição para mudança.\')\n cols=[\'age\',\'role\',\'income\',\'housing\',\'skill\',\'mode\'];X=df[cols]\n pre=ColumnTransformer([(\'numeric\',Pipeline([(\'impute\',SimpleImputer(strategy=\'median\',keep_empty_features=True,add_indicator=True)),(\'scale\',StandardScaler())]),[\'age\']),(\'categories\',Pipeline([(\'impute\',SimpleImputer(strategy=\'constant\',fill_value=\'Sem resposta\')),(\'encode\',OneHotEncoder(handle_unknown=\'ignore\',sparse_output=False))]),cols[1:])])\n models={\'Referência de prevalência\':DummyClassifier(strategy=\'prior\'),\'Regressão logística\':LogisticRegression(C=1,max_iter=1500,random_state=SEED),\'Random forest\':RandomForestClassifier(n_estimators=100,max_depth=4,min_samples_leaf=10,random_state=SEED,n_jobs=1)}\n folds=list(StratifiedKFold(5,shuffle=True,random_state=SEED).split(X,y));out=[]\n for name,model in models.items():\n  pred=np.zeros(len(df));scores=[]\n  for train,test in folds:\n   pipe=Pipeline([(\'preparation\',pre),(\'model\',model)]);pipe.fit(X.iloc[train],y[train]);p=pipe.predict_proba(X.iloc[test])[:,1];pred[test]=p;scores.append(balanced_accuracy_score(y[test],p>=.5))\n  out.append(dict(model=name,n=len(df),balanced_accuracy=balanced_accuracy_score(y,pred>=.5),auc=roc_auc_score(y,pred),brier=brier_score_loss(y,pred),log_loss=log_loss(y,pred),fold_balanced_accuracy=scores))\n return dict(target=\'qChange: Sim versus Não; disposição autorrelatada, não mudança observada\',n=len(df),excluded=len(records)-len(df),class_counts=counts.to_dict(),models=out,interpretation=\'Compare cada modelo com a referência de prevalência. Desempenho em validação interna não demonstra transportabilidade, causalidade ou utilidade operacional. Hiperparâmetros fixados; sem busca no conjunto de teste. qChance, qBarriers e cenários não entram nos preditores para evitar dependência do desfecho.\')\n\ndef scenarios(records):\n rows=[]\n for r in records:\n  a=r[\'answers\'];current=a.get(\'q16\')\n  if not current:continue\n  for i in range(1,7):\n   value=a.get(\'qSituation\'+str(i))\n   if value is not None:rows.append(dict(id=r[\'response_id\'],scenario=str(i),change=int(not value.startswith(\'Continuaria\') and value!=current)))\n df=pd.DataFrame(rows)\n if df.empty or df.id.nunique()<40 or df.change.nunique()<2:raise ValueError(\'Cenários requerem 40 participantes com respostas e variação nas escolhas.\')\n counts=df.groupby(\'scenario\').change.agg([\'count\',\'sum\']);\n if len(counts)<6 or (counts[\'sum\']<5).any() or ((counts[\'count\']-counts[\'sum\'])<5).any():raise ValueError(\'Cada cenário precisa de pelo menos cinco respostas em cada categoria binária.\')\n fit=smf.gee(\'change ~ C(scenario)\',groups=\'id\',data=df,family=sm.families.Binomial(),cov_struct=sm.cov_struct.Exchangeable()).fit()\n if not getattr(fit,\'converged\',True):raise ValueError(\'GEE não convergiu.\')\n ci=fit.conf_int();terms=[dict(term=k,log_odds=fit.params[k],odds_ratio=np.exp(fit.params[k]),ci_low=np.exp(ci.loc[k,0]),ci_high=np.exp(ci.loc[k,1]),p=fit.pvalues[k]) for k in fit.params.index]\n return dict(participants=df.id.nunique(),observations=len(df),terms=terms,by_scenario=counts.reset_index().to_dict(\'records\'),interpretation=\'Contrastes associativos entre situações, em relação à situação 1, com correlação das respostas da mesma pessoa e variância robusta. A escolha foi reduzida a mudança versus manutenção; a distribuição original por alternativa deve acompanhar o resultado. Não são efeitos causais nem elasticidades.\')\n\ndef moderation(records):\n df=frame(records).dropna(subset=[\'target\',\'age\',\'skill\',\'mode\']);df[\'knows_cycle\']=(~df.skill.str.startswith(\'Não\')).astype(int);df[\'active_mode\']=df[\'mode\'].isin([\'A pé\',\'Bicicleta comum\',\'Bicicleta ou patinete elétrico\']).astype(int);df[\'age10\']=(df.age-30)/10\n counts=df.groupby([\'knows_cycle\',\'active_mode\',\'target\']).size()\n if len(df)<100 or len(counts)!=8 or counts.min()<5:raise ValueError(\'Interação requer 100 casos completos e pelo menos cinco casos por célula habilidade × modo × desfecho.\')\n fit=smf.logit(\'target ~ knows_cycle * active_mode + age10\',data=df).fit(disp=False,maxiter=150)\n if not fit.mle_retvals.get(\'converged\'):raise ValueError(\'Modelo de interação não convergiu.\')\n ci=fit.conf_int();terms=[dict(term=k,odds_ratio=np.exp(fit.params[k]),ci_low=np.exp(ci.loc[k,0]),ci_high=np.exp(ci.loc[k,1]),p=fit.pvalues[k]) for k in fit.params.index]\n return dict(n=len(df),excluded=len(records)-len(df),terms=terms,interpretation=\'Interação exploratória na escala logit entre saber pedalar (q10) e modo ativo atual, ajustada apenas por idade. q10 combina habilidade e frequência; não mede uma competência latente validada. A interação não prova moderação causal. Confundimento por vínculo e renda ainda precisa ser avaliado no plano final.\')\n\ndef run(payload):\n records=payload.get(\'records\',[])\n if not isinstance(records,list) or not 1<=len(records)<=2000:raise ValueError(\'Selecione entre 1 e 2.000 registros.\')\n if any(r.get(\'schema_version\')!=VERSION or not isinstance(r.get(\'answers\'),dict) for r in records):raise ValueError(\'Use exclusivamente o instrumento \'+VERSION)\n ids=[r.get(\'response_id\') for r in records]\n if any(not isinstance(i,str) or not i for i in ids) or len(set(ids))!=len(ids):raise ValueError(\'Identificadores inválidos ou duplicados.\')\n if any(r.get(\'is_demo\') not in [True,False] or r.get(\'is_synthetic\') not in [True,False] or (r.get(\'is_synthetic\') and not r.get(\'is_demo\')) for r in records):raise ValueError(\'Tipo de base inválido.\')\n kinds={(\'synthetic\' if r.get(\'is_synthetic\') else \'demo\' if r.get(\'is_demo\') else \'real\') for r in records}\n if len(kinds)!=1:raise ValueError(\'Separe simulação, teste e coleta real.\')\n if any(r[\'answers\'].get(\'q1\')!=\'Sim\' or not isinstance(r[\'answers\'].get(\'q2\'),int) or not 18<=r[\'answers\'][\'q2\']<=120 for r in records):raise ValueError(\'Verifique consentimento e idade.\')\n schema=json.loads((Path(__file__).resolve().parent.parent/\'public\'/\'questionario-v4.schema.json\').read_text(encoding=\'utf-8-sig\'))\n for r in records:\n  a=r[\'answers\']\n  for q in schema[\'questions\']:\n   v=a.get(q[\'id\']);condition=q.get(\'when\')\n   if condition:\n    parent=a.get(condition[\'id\']);visible=isinstance(parent,str) and parent.startswith(condition[\'prefix\']) if \'prefix\' in condition else parent==condition.get(\'value\')\n    if not visible and v is not None:raise ValueError(\'Resposta condicional incompatível: \'+q[\'id\'])\n   if v is None:continue\n   if q[\'type\']==\'radio\' and v not in q[\'options\']:raise ValueError(\'Alternativa inválida: \'+q[\'id\'])\n   if q[\'type\']==\'multi\' and (not isinstance(v,list) or any(x not in q[\'options\'] for x in v) or len(v)!=len(set(v)) or len(v)>q.get(\'limit\',len(q[\'options\']))):raise ValueError(\'Seleção inválida: \'+q[\'id\'])\n   if q[\'type\']==\'number\' and (not isinstance(v,int) or isinstance(v,bool) or v<q.get(\'min\',0) or v>q.get(\'max\',1000000)):raise ValueError(\'Número inválido: \'+q[\'id\'])\n method=payload.get(\'method\');methods={\'ml\':ml,\'scenarios\':scenarios,\'moderation\':moderation}\n if method not in methods:raise ValueError(\'Método indisponível. Mediação causal não é identificada pelo desenho atual.\')\n result=methods[method](records)\n return clean(dict(method=method,version=VERSION,kind=next(iter(kinds)),seed=SEED,result=result,fingerprint=hashlib.sha256(json.dumps(records,sort_keys=True,ensure_ascii=False).encode()).hexdigest(),warning=\'DADOS FICTÍCIOS: resultados exclusivos para demonstração.\' if kinds=={\'synthetic\'} else \'Associações da amostra; não implicam causalidade ou representatividade.\'))\n\nclass handler(BaseHTTPRequestHandler):\n def do_POST(self):\n  try:\n   if self.headers.get(\'Origin\',\'\').split(\'://\')[-1]!=self.headers.get(\'Host\'):return self.reply(403,{\'error\':\'Origem não autorizada.\'})\n   size=int(self.headers.get(\'Content-Length\',0))\n   if size>3500000:return self.reply(413,{\'error\':\'Base muito extensa.\'})\n   p=json.loads(self.rfile.read(size));records=p.get(\'records\',[])\n   if os.environ.get(\'COLLECTION_RELEASED\')==\'true\' or any(not r.get(\'is_demo\') for r in records):\n    cookies=dict(x.strip().split(\'=\',1) for x in self.headers.get(\'Cookie\',\'\').split(\';\') if \'=\' in x);body,sig=cookies.get(\'ufv_admin\',\'\').split(\'.\')\n    secret=os.environ.get(\'SESSION_SECRET\',\'\')\n    if len(secret)<32:raise ValueError(\'Sessão não configurada.\')\n    expected=base64.urlsafe_b64encode(hmac.new(secret.encode(),body.encode(),hashlib.sha256).digest()).decode().rstrip(\'=\')\n    data=json.loads(base64.urlsafe_b64decode(body+\'=\'*(-len(body)%4)))\n    if not hmac.compare_digest(sig,expected) or data.get(\'exp\',0)<time.time()*1000 or data.get(\'role\')!=\'admin\':return self.reply(401,{\'error\':\'Entre novamente na administração.\'})\n   self.reply(200,run(p))\n  except ValueError as e:self.reply(400,{\'error\':str(e)})\n  except FileNotFoundError:self.reply(503,{\'error\':\'O esquema do questionário não foi incluído no ambiente de análise.\'})\n  except Exception as e:\n   print(json.dumps({\'analysis_error_type\':type(e).__name__}),flush=True)\n   self.reply(422,{\'error\':\'O ajuste falhou. Verifique casos válidos, variação e categorias do recorte.\'})\n def reply(self,status,value):\n  self.send_response(status);self.send_header(\'Content-Type\',\'application/json; charset=utf-8\');self.send_header(\'Cache-Control\',\'no-store\');self.end_headers();self.wfile.write(json.dumps(value,ensure_ascii=False,allow_nan=False).encode())\n def log_message(self,*args):pass\nif __name__==\'__main__\':\n import sys\n print(json.dumps(run(json.load(sys.stdin)),ensure_ascii=False,allow_nan=False))\n',encoding='utf-8')
Path('public/questionario-v4.schema.json').write_text('{\n  "version": "ufv-mobilidade-4.2.0",\n  "design_version": "rotina-mapa-oficial-revisao-2026-10-10",\n  "title": "Pesquisa sobre deslocamentos até o campus da UFV – Viçosa",\n  "source": "Questionario_UFV_sugestoes.docx",\n  "questions": [\n    {\n      "id": "q1",\n      "number": 1,\n      "title": "Li o Termo de Consentimento Livre e Esclarecido e aceito participar.",\n      "type": "radio",\n      "options": [\n        "Sim",\n        "Não"\n      ]\n    },\n    {\n      "id": "q2",\n      "number": 2,\n      "title": "Qual a sua idade, em anos completos?",\n      "type": "number",\n      "options": [],\n      "min": 0,\n      "step": 1,\n      "max": 120\n    },\n    {\n      "id": "q3",\n      "number": 3,\n      "title": "Qual o seu vínculo com a UFV?",\n      "type": "radio",\n      "options": [\n        "Estudante de graduação",\n        "Estudante de pós-graduação",\n        "Docente",\n        "Técnico-administrativo",\n        "Terceirizado ou funcionário de fundação"\n      ]\n    },\n    {\n      "id": "q3a",\n      "number": "3a",\n      "title": "Qual é o nível de escolaridade mais alto que você concluiu?",\n      "type": "radio",\n      "options": [\n        "Ensino médio",\n        "Graduação",\n        "Especialização",\n        "Mestrado",\n        "Doutorado"\n      ]\n    },\n    {\n      "id": "q4",\n      "number": 4,\n      "title": "Qual o seu gênero?",\n      "type": "radio",\n      "options": [\n        "Feminino",\n        "Masculino",\n        "Outro",\n        "Prefiro não informar"\n      ]\n    },\n    {\n      "id": "q5",\n      "number": 5,\n      "title": "Qual é a renda mensal total das pessoas que compartilham suas despesas?",\n      "type": "radio",\n      "options": [\n        "Até 1 SM (até R$ 1.621)",\n        "De 1 a 3 SM (R$ 1.621,01 a R$ 4.863)",\n        "De 3 a 5 SM (R$ 4.863,01 a R$ 8.105)",\n        "De 5 a 10 SM (R$ 8.105,01 a R$ 16.210)",\n        "Acima de 10 SM (acima de R$ 16.210)",\n        "Prefiro não informar"\n      ],\n      "hint": "Em repúblicas, inclua somente quem compartilha seu orçamento; não some automaticamente a renda de todos os moradores. Valores conforme o questionário revisado."\n    },\n    {\n      "id": "q6",\n      "number": 6,\n      "title": "Quantas pessoas moram no seu domicílio, contando você?",\n      "type": "number",\n      "options": [],\n      "min": 1,\n      "step": 1\n    },\n    {\n      "id": "qHousing",\n      "number": "Housing",\n      "title": "Onde você mora durante o período letivo?",\n      "type": "radio",\n      "options": [\n        "República ou casa dividida com estudantes",\n        "Sozinho(a)",\n        "Com família",\n        "Pensionato ou quarto alugado",\n        "Alojamento da UFV"\n      ]\n    },\n    {\n      "id": "q8",\n      "number": 8,\n      "title": "Você tem carteira de motorista (CNH)?",\n      "type": "radio",\n      "options": [\n        "Não",\n        "Sim, de carro",\n        "Sim, de moto",\n        "Sim, de carro e moto"\n      ]\n    },\n    {\n      "id": "q9",\n      "number": "9",\n      "title": "Quais destes você tem à disposição para usar no dia a dia?",\n      "type": "multi",\n      "options": [\n        "Carro",\n        "Moto",\n        "Bicicleta comum",\n        "Bicicleta ou patinete elétrico",\n        "Nenhum"\n      ],\n      "exclusive": [\n        "Nenhum"\n      ]\n    },\n    {\n      "id": "q10",\n      "number": "10",\n      "title": "Você sabe andar de bicicleta?",\n      "type": "radio",\n      "options": [\n        "Sim, e ando com alguma frequência",\n        "Sim, mas raramente ou nunca ando",\n        "Não sei andar de bicicleta"\n      ]\n    },\n    {\n      "id": "q11",\n      "number": 11,\n      "title": "De onde você costuma sair para ir ao campus?",\n      "type": "location",\n      "options": [],\n      "hint": "Marque a esquina ou quadra mais próxima. Não informe número de imóvel. Se preferir, responda pelos campos de texto."\n    },\n    {\n      "id": "q12",\n      "number": 12,\n      "title": "Quantos dias por semana você vem ao campus?",\n      "type": "radio",\n      "options": [\n        "1",\n        "2",\n        "3",\n        "4",\n        "5",\n        "6 ou 7"\n      ]\n    },\n    {\n      "id": "q13",\n      "number": "13",\n      "title": "A que horas você costuma chegar ao campus?",\n      "type": "radio",\n      "options": [\n        "Antes das 7h",\n        "Entre 7h e 8h59",\n        "Entre 9h e 12h59",\n        "Entre 13h e 17h59",\n        "A partir das 18h"\n      ],\n      "hint": "Considere a maioria dos dias. Se vai mais de uma vez no mesmo dia, marque a primeira chegada."\n    },\n    {\n      "id": "qDeparture",\n      "number": "Departure",\n      "title": "A que horas você costuma sair do campus?",\n      "type": "radio",\n      "options": [\n        "Antes das 12h",\n        "Entre 12h e 17h59",\n        "Entre 18h e 19h59",\n        "Entre 20h e 21h59",\n        "A partir das 22h"\n      ],\n      "hint": "Considere a maioria dos dias. Se sai mais de uma vez no mesmo dia, marque a última saída."\n    },\n    {\n      "id": "q15",\n      "number": 15,\n      "title": "Em qual prédio ou local do campus você chega primeiro?",\n      "type": "text",\n      "options": [],\n      "hint": "Você pode buscar o nome do local ou marcar no mapa oficial. Não precisa saber o número do prédio."\n    },\n    {\n      "id": "q16",\n      "number": "16",\n      "title": "Qual meio de transporte você usa para ir ao campus?",\n      "type": "radio",\n      "options": [\n        "A pé",\n        "Bicicleta comum",\n        "Bicicleta ou patinete elétrico",\n        "Moto",\n        "Carro",\n        "Ônibus",\n        "Carona",\n        "Carro de aplicativo ou táxi"\n      ],\n      "hint": "Se usar mais de um meio, marque o do trecho mais longo."\n    },\n    {\n      "id": "q20",\n      "number": "20",\n      "title": "De modo geral, o que você mais considera ao escolher como ir ao campus?",\n      "type": "multi",\n      "options": [\n        "Rapidez e pontualidade (tempo de viagem, chegar no horário)",\n        "Custo",\n        "Segurança",\n        "Conforto e esforço (não suar, não se molhar, evitar subidas)",\n        "Saúde e meio ambiente (fazer exercício, poluir menos)"\n      ],\n      "limit": 2\n    },\n    {\n      "id": "q22",\n      "number": "22",\n      "title": "Como costuma se deslocar dentro do campus (entre prédios, restaurante)?",\n      "type": "radio",\n      "options": [\n        "A pé",\n        "Bicicleta",\n        "Bicicleta ou patinete elétrico",\n        "Moto",\n        "Carro",\n        "Ônibus",\n        "Não me desloco"\n      ]\n    },\n    {\n      "id": "q22a",\n      "number": "22a",\n      "title": "Qual meio de transporte você usa para ir embora do campus?",\n      "type": "radio",\n      "options": [\n        "Pelo mesmo meio de transporte da ida",\n        "Por outro meio de transporte"\n      ]\n    },\n    {\n      "id": "qReturnMode",\n      "number": "ReturnMode",\n      "title": "Qual meio de transporte você usa na volta?",\n      "type": "radio",\n      "options": [\n        "A pé",\n        "Bicicleta comum",\n        "Bicicleta ou patinete elétrico",\n        "Moto",\n        "Carro",\n        "Ônibus",\n        "Carona",\n        "Carro de aplicativo ou táxi"\n      ],\n      "hint": "Se usar mais de um meio, marque o do trecho mais longo.",\n      "when": {\n        "id": "q22a",\n        "value": "Por outro meio de transporte"\n      }\n    },\n    {\n      "id": "qReturnReason",\n      "number": "ReturnReason",\n      "title": "Por que você volta por um meio diferente do da ida?",\n      "type": "multi",\n      "options": [\n        "O relevo é diferente nos dois sentidos (um deles tem mais subida)",\n        "O horário de ida e o de volta são diferentes (pressa para chegar, volta à noite, cansaço no fim do dia)",\n        "O ônibus (horário ou frequência) atende só um dos sentidos",\n        "Tenho carona ou veículo disponível só na ida ou só na volta",\n        "Faço outra parada em um dos trajetos (trabalho, compras, buscar alguém etc.)",\n        "Outro"\n      ],\n      "limit": 2,\n      "other": true,\n      "when": {\n        "id": "q22a",\n        "value": "Por outro meio de transporte"\n      }\n    },\n    {\n      "id": "qChange",\n      "number": "Change",\n      "title": "Você estaria disposto(a) a trocar seu meio atual por um destes para ir e voltar do campus na maioria dos dias?",\n      "type": "radio",\n      "options": [\n        "Não, continuaria com meu meio atual",\n        "Sim, passaria a ir a pé",\n        "Sim, passaria a usar bicicleta comum própria",\n        "Sim, passaria a usar bicicleta ou patinete elétrico próprio"\n      ],\n      "hint": "Considere uma mudança em relação ao meio que você usa hoje. Se já utiliza uma dessas opções, você pode escolher outra; se pretende manter seu meio atual, marque “Não”."\n    },\n    {\n      "id": "qChance",\n      "number": "Chance",\n      "title": "Qual a chance de você realmente fazer essa mudança nos próximos 6 meses?",\n      "type": "radio",\n      "options": [\n        "Nenhuma",\n        "Pequena",\n        "Média",\n        "Grande",\n        "Já estou fazendo"\n      ],\n      "when": {\n        "id": "qChange",\n        "prefix": "Sim,"\n      }\n    },\n    {\n      "id": "qBarriers",\n      "number": "Barriers",\n      "title": "O que mais impede você de fazer essa mudança?",\n      "type": "multi",\n      "options": [\n        "Subidas no caminho (ladeiras, relevo)",\n        "Esforço e clima (distância, condição física, calor, suor, chuva)",\n        "Segurança (trânsito e pessoal)",\n        "Rotina (levar material ou outra pessoa, horários e compromissos)",\n        "Custo de comprar e manter uma bicicleta"\n      ],\n      "when": {\n        "id": "qChange",\n        "value": "Não, continuaria com meu meio atual"\n      },\n      "hint": "Marque os impedimentos que se aplicam a você. Se não houver, pode deixar sem resposta."\n    },\n    {\n      "id": "qSituation1",\n      "number": "Situation1",\n      "title": "Nessa situação, como você iria e voltaria do campus na maioria dos dias?",\n      "type": "radio",\n      "options": [\n        "Continuaria com meu meio atual",\n        "A pé",\n        "Bicicleta comum",\n        "Bicicleta ou patinete elétrico"\n      ],\n      "situation": "Ciclovia ou ciclofaixa contínua em toda a sua rota até o campus"\n    },\n    {\n      "id": "qSituation2",\n      "number": "Situation2",\n      "title": "Nessa situação, como você iria e voltaria do campus na maioria dos dias?",\n      "type": "radio",\n      "options": [\n        "Continuaria com meu meio atual",\n        "A pé",\n        "Bicicleta comum",\n        "Bicicleta ou patinete elétrico"\n      ],\n      "situation": "Calçadas largas, contínuas e sem obstáculos em toda a sua rota até o campus"\n    },\n    {\n      "id": "qSituation3",\n      "number": "Situation3",\n      "title": "Nessa situação, como você iria e voltaria do campus na maioria dos dias?",\n      "type": "radio",\n      "options": [\n        "Continuaria com meu meio atual",\n        "A pé",\n        "Bicicleta comum",\n        "Bicicleta ou patinete elétrico"\n      ],\n      "situation": "Infraestrutura no prédio de destino (bicicletário seguro com recarga e vestiário)"\n    },\n    {\n      "id": "qSituation4",\n      "number": "Situation4",\n      "title": "Nessa situação, como você iria e voltaria do campus na maioria dos dias?",\n      "type": "radio",\n      "options": [\n        "Continuaria com meu meio atual",\n        "A pé",\n        "Bicicleta comum",\n        "Bicicleta ou patinete elétrico"\n      ],\n      "situation": "Bicicletas comuns e elétricas compartilhadas pela UFV, com estações no campus e nos bairros, para pegar num lugar e deixar no outro, com aluguel subsidiado"\n    },\n    {\n      "id": "qSituation5",\n      "number": "Situation5",\n      "title": "Nessa situação, como você iria e voltaria do campus na maioria dos dias?",\n      "type": "radio",\n      "options": [\n        "Continuaria com meu meio atual",\n        "A pé",\n        "Bicicleta comum",\n        "Bicicleta ou patinete elétrico"\n      ],\n      "situation": "O estacionamento no campus passa a ser cobrado"\n    },\n    {\n      "id": "qSituation6",\n      "number": "Situation6",\n      "title": "Nessa situação, como você iria e voltaria do campus na maioria dos dias?",\n      "type": "radio",\n      "options": [\n        "Continuaria com meu meio atual",\n        "A pé",\n        "Bicicleta comum",\n        "Bicicleta ou patinete elétrico"\n      ],\n      "situation": "A passagem de ônibus ou o preço do combustível sofre um aumento"\n    },\n    {\n      "id": "qEvaluation",\n      "number": "4",\n      "title": "Pensando no seu caminho entre a casa e o campus, como você avalia cada meio de transporte?",\n      "type": "matrix",\n      "options": [],\n      "modes": [\n        "Meio atual da ida",\n        "A pé",\n        "Bicicleta comum",\n        "Bicicleta ou patinete elétrico"\n      ],\n      "items": [\n        "Segurança (risco de acidente, assalto ou furto)",\n        "Conforto (pouco esforço, pouco calor e suor, protegido da chuva)",\n        "Praticidade (chegar no horário, sair quando quiser, carregar material)"\n      ],\n      "ends": [\n        "Muito ruim",\n        "Muito bom"\n      ],\n      "hint": "Dê uma nota de 1 a 5. Se não souber, marque “Não sei avaliar”. Sua opinião importa mesmo para os meios que você não usa."\n    }\n  ]\n}',encoding='utf-8')
spec=importlib.util.spec_from_file_location('ufv_analysis','api/analysis_v4.py')
analysis=importlib.util.module_from_spec(spec);spec.loader.exec_module(analysis)
print(Path('api/analysis_v4.py').read_text())

## Recorte e descrição
Período em horário de Brasília. Ausências são contadas por variável e não convertidas em não. Percentuais em questões múltiplas usam respondentes válidos como denominador.

In [ ]:
ROLE = None
MODE = None
DATE_FROM = None  # exemplo '2026-09-01'
DATE_UNTIL = None
selected=[]
for r in records:
    day=pd.Timestamp(r['completed_at']).tz_convert('America/Sao_Paulo').strftime('%Y-%m-%d')
    if ROLE and r['answers'].get('q3')!=ROLE: continue
    if MODE and r['answers'].get('q16')!=MODE: continue
    if DATE_FROM and day<DATE_FROM: continue
    if DATE_UNTIL and day>DATE_UNTIL: continue
    selected.append(r)
if not selected: raise ValueError('Recorte vazio.')
frame=pd.DataFrame([r['answers'] for r in selected])
schema=json.loads(Path('public/questionario-v4.schema.json').read_text())
summary=[]
for q in schema['questions']:
    if q['type'] not in ['radio','multi'] or q['id']=='q1': continue
    vals=[r['answers'].get(q['id']) for r in selected]
    valid=[v for v in vals if v is not None]
    items=[x for v in valid for x in (v if isinstance(v,list) else [v])]
    for label,count in pd.Series(items,dtype='object').value_counts().items():
        summary.append({'question':q['id'],'answer':label,'n':int(count),'valid':len(valid),'missing':len(vals)-len(valid),'percent':100*count/len(valid)})
summary=pd.DataFrame(summary)
display(summary.head(20))
summary.to_csv(OUTPUT/'descritivas.csv',index=False)


## Ajustes e critérios de elegibilidade
Modelos podem recusar recortes pequenos ou sem variação. Essa recusa é um resultado de adequação, não deve ser contornada mudando dados. Os intervalos são associativos; a demonstração sintética não testa hipóteses populacionais.

In [ ]:
results={}
for method in ['ml','scenarios','moderation']:
    try:
        results[method]=analysis.run({'method':method,'records':selected})
        print(method,results[method]['warning'])
        print(results[method]['result']['interpretation'])
        display(pd.DataFrame(results[method]['result'].get('models',results[method]['result'].get('terms',[]))))
    except ValueError as e:
        results[method]={'status':'not_estimable','reason':str(e)}
        print(method,str(e))
(OUTPUT/'modelos.json').write_text(json.dumps(results,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')


## Gráficos e rede agregada
Os gráficos representam somente o recorte. Não exportar células raras de dados reais. Ligações na rede significam contagens de respostas, não trajetos percorridos.

In [ ]:
import matplotlib.pyplot as plt, networkx as nx
plt.rcParams.update({'figure.dpi':140,'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
counts=frame['q16'].dropna().value_counts().sort_values()
fig,ax=plt.subplots(figsize=(10,5));counts.plot.barh(ax=ax,color='#901812');ax.set_xlabel('Respostas');ax.set_ylabel('');ax.set_title('Meio de ida — '+next(iter(kinds)));fig.tight_layout();fig.savefig(OUTPUT/'modos.png');plt.show()
pairs=frame.dropna(subset=['q16','q15']).groupby(['q16','q15']).size()
pairs=pairs[pairs>=5].sort_values(ascending=False).head(30)
G=nx.Graph()
for (mode,destination),n in pairs.items(): G.add_edge('M: '+mode,'D: '+destination,weight=int(n))
if G.number_of_edges():
    fig,ax=plt.subplots(figsize=(14,9));pos=nx.spring_layout(G,seed=8421264)
    nx.draw_networkx(G,pos,ax=ax,font_size=7,node_color=['#d6a59d' if n.startswith('M:') else '#b5ccae' for n in G],width=[G[u][v]['weight']**.5 for u,v in G.edges],node_size=650)
    ax.set_title('Rede agregada de meios e destinos — '+next(iter(kinds)));ax.axis('off');fig.tight_layout();fig.savefig(OUTPUT/'rede.png');plt.show()


## Mapa real de Viçosa
A cartografia é pública; as origens da demonstração são fictícias. Grupos com menos de cinco observações são omitidos. Não projetar marcações da imagem do campus em latitude/longitude sem georreferenciamento.

In [ ]:
import folium
m=folium.Map(location=[-20.755,-42.875],zoom_start=14,tiles=None)
for name in ['vicosa-streets.geojson','ufv-campus.geojson']:
    with urllib.request.urlopen(BASE+name) as response: geo=json.load(response)
    folium.GeoJson(geo,name=name,style_function=lambda feature: {'color':'#77796b','weight':1.5,'fillOpacity':.08}).add_to(m)
points=[]
for r in selected:
    p=(r['answers'].get('q11') or {}).get('point')
    if p: points.append((p['lat'],p['lng']))
for (lat,lng),n in pd.Series(points,dtype='object').value_counts().items():
    if n>=5: folium.CircleMarker([lat,lng],radius=min(22,5+n**.5),color='#901812',fill=True,tooltip=f'{n} respostas agregadas — {next(iter(kinds))}').add_to(m)
folium.LayerControl().add_to(m)
m.save(str(OUTPUT/'mapa.html'));display(m)


## Manifesto, limitações e exportação
A base analisada é identificada por hash, sem incluir microdados no manifesto. Dados fictícios não demonstram desempenho populacional. Não estimamos mediação, disposição a pagar, elasticidades ou rotas ótimas. O artigo requer discussão humana e atualização das referências antes de submissão.

In [ ]:
import importlib.metadata, datetime
manifest={'version':'ufv-mobilidade-4.2.0','n':len(selected),'kind':next(iter(kinds)),'seed':8421264,'generated_at':datetime.datetime.now(datetime.timezone.utc).isoformat(),'fingerprint':hashlib.sha256(json.dumps(selected,sort_keys=True,ensure_ascii=False).encode()).hexdigest(),'filters':{'role':ROLE,'mode':MODE,'from':DATE_FROM,'until':DATE_UNTIL},'packages':{p:importlib.metadata.version(p) for p in ['numpy','pandas','scipy','scikit-learn','statsmodels']}}
(OUTPUT/'manifesto.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print('Arquivos gravados em',OUTPUT)
# A exportação é uma ação deliberada. Revise células pequenas antes de compartilhar resultados reais.
EXPORT = False
if EXPORT:
    import shutil
    from google.colab import files
    shutil.make_archive('resultados_ufv_v4','zip',OUTPUT)
    files.download('resultados_ufv_v4.zip')
